In [ ]:
import os
import re
import glob

# ── CONFIG ──────────────────────────────────────────
FOLDER_PATH    = r"D:\OneDrive - Personal\Desktop\mms-mon-finetuned\Kitchen\Final"  # <-- change this
STRIP_COMMENTS = True   # set False to keep # comment lines
MASK_KEYS      = True   # set False to skip masking values
MIN_LENGTH     = 7      # values longer than this get masked
DRY_RUN        = False  # set True to preview without saving
# ────────────────────────────────────────────────────

SKIP_VALUE = re.compile(r'^\d+\.?\d*$|^(true|false)$', re.IGNORECASE)

def mask_env_files():
    patterns = [".env", ".env_local", ".env.*"]
    files = []

    for root, dirs, filenames in os.walk(FOLDER_PATH):
        for filename in filenames:
            for pattern in patterns:
                if filename == pattern or (pattern.endswith(".*") and filename.startswith(pattern[:-1])):
                    files.append(os.path.join(root, filename))

    files = list(set(files))

    if not files:
        print(f"No .env files found in: {FOLDER_PATH}")
        return

    print(f"Found {len(files)} file(s):\n")

    for path in files:
        with open(path, "r") as f:
            lines = f.readlines()

        new_lines = []
        changed = False

        for line in lines:
            stripped = line.strip()

            # Delete comment lines
            if STRIP_COMMENTS and stripped.startswith("#"):
                changed = True
                continue

            # Mask long values
            if MASK_KEYS and "=" in stripped and not stripped.startswith("#"):
                key, _, val = stripped.partition("=")
                val = val.strip()
                if len(val) > MIN_LENGTH and not SKIP_VALUE.match(val):
                    new_lines.append(f"{key.strip()}={val[:2]}...\n")
                    changed = True
                    continue

            new_lines.append(line)

        if changed:
            if DRY_RUN:
                print(f"[DRY RUN] Would update: {path}")
            else:
                with open(path, "w") as f:
                    f.writelines(new_lines)
                print(f"Updated: {path}")
        else:
            print(f"No changes: {path}")

mask_env_files()

Found 5 file(s):

Updated: D:\OneDrive - Personal\Desktop\mms-mon-finetuned\Kitchen\Final\.env.example
Updated: D:\OneDrive - Personal\Desktop\mms-mon-finetuned\Kitchen\Final\browser-use-ui\.env
Updated: D:\OneDrive - Personal\Desktop\mms-mon-finetuned\Kitchen\Final\browser-use-ui\.env.browser-use.example
Updated: D:\OneDrive - Personal\Desktop\mms-mon-finetuned\Kitchen\Final\.env
Updated: D:\OneDrive - Personal\Desktop\mms-mon-finetuned\Kitchen\Final\browser-use-ui\.env.example
